In [3]:
from langgraph.graph import StateGraph, START, END
from langchain_groq import ChatGroq
from typing import TypedDict
from dotenv import load_dotenv




In [4]:
load_dotenv()

True

In [24]:
model = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0
)


In [19]:
# create a state

class LLMState(TypedDict):

    question: str
    answer: str

In [20]:
def llm_qa(state: LLMState) -> LLMState:

    # extract the question from state
    question = state['question']

    # form a prompt
    prompt = f'Answer the following question {question}'

    # ask that question to the LLM
    answer = model.invoke(prompt).content

    # update the answer in the state
    state['answer'] = answer

    return state

In [21]:
# create our graph

graph = StateGraph(LLMState)

# add nodes
graph.add_node('llm_qa', llm_qa)

# add edges
graph.add_edge(START, 'llm_qa')
graph.add_edge('llm_qa', END)

# compile
workflow = graph.compile()

In [22]:
# execute

intial_state = {'question': 'How far is moon from the earth?'}

final_state = workflow.invoke(intial_state)

print(final_state['answer'])

The Moon is, on average, about **384 400 kilometers** (≈ 238 900 miles) from Earth.  

- **Perigee (closest point):** ~ 363 300 km  
- **Apogee (farthest point):** ~ 405 500 km  

So the distance varies by roughly ± 21 % over each lunar orbit.


In [23]:
model.invoke('How far is moon from the earth?').content

'The Moon is, on average, about **384\u202f400\u202fkm** (≈\u202f238\u202f855\u202fmi) from Earth.  \nBecause the Moon’s orbit is slightly elliptical, the distance varies:\n\n| Point in orbit | Distance from Earth |\n|----------------|---------------------|\n| **Perigee** (closest) | ~\u202f363\u202f300\u202fkm (≈\u202f225\u202f700\u202fmi) |\n| **Apogee** (farthest) | ~\u202f405\u202f500\u202fkm (≈\u202f251\u202f900\u202fmi) |\n\nSo the range is roughly **363\u202f000\u202f–\u202f406\u202f000\u202fkm**.  \nIn terms of light travel time, it takes about **1.28\u202fseconds** for light to cross that distance.'